# AirShift — XGBoost Model Tuning

This notebook focuses on improving the XGBoost model selected during the model comparison stage.

The tuning process will explore different XGBoost hyperparameters and compare the tuned model with the baseline XGBoost model.

The final test period will remain completely unseen during tuning and will only be used for the final evaluation.


## 1. Load the Labeled Dataset

The labeled dataset is loaded from the processed data directory.

This dataset contains the engineered features and the binary deterioration target created in the previous stages of the AirShift pipeline.


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)
import time
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import joblib

In [4]:
DATA_PATH = Path("../data/processed/airshift_labeled.csv")

df = pd.read_csv(DATA_PATH)

df["datetime"] = pd.to_datetime(df["datetime"])

print("Dataset shape:", df.shape)
print("Number of stations:", df["station"].nunique())
print("Date range:", df["datetime"].min(), "to", df["datetime"].max())

Dataset shape: (418381, 100)
Number of stations: 12
Date range: 2013-03-01 00:00:00 to 2017-02-28 17:00:00


In [ ]:
import joblib
from pathlib import Path

MODEL_PATH = Path("../models/xgboost_final.joblib")

loaded_model = joblib.load(MODEL_PATH)

print("Model loaded successfully.")
print("Preprocessor fitted:",
      hasattr(loaded_model.named_steps["preprocessor"], "transformers_"))
print("XGBoost fitted:",
      hasattr(loaded_model.named_steps["model"], "n_features_in_"))

Model loaded successfully.
Preprocessor fitted: True
XGBoost fitted: False


In [9]:
print("\nTarget distribution:")
print(df["Deterioration"].value_counts())

print("\nTarget proportions:")
print(df["Deterioration"].value_counts(normalize=True).round(4))


Target distribution:
Deterioration
0.0    219985
1.0    198396
Name: count, dtype: int64

Target proportions:
Deterioration
0.0    0.5258
1.0    0.4742
Name: proportion, dtype: float64


In [10]:
# Quick validation
print("Missing target values:", df["Deterioration"].isna().sum())
print("Duplicate rows:", df.duplicated().sum())

Missing target values: 0
Duplicate rows: 0


## 2. Define Features and Target

The deterioration label is used as the target variable, while identifier and datetime columns are excluded from the model features.

The `datetime` column is retained separately for creating the chronological data splits.


In [11]:
TARGET = "Deterioration"

EXCLUDED_COLUMNS = [
    "Deterioration",
    "No",
    "datetime"
]

X = df.drop(columns=EXCLUDED_COLUMNS)
y = df[TARGET]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Number of features:", X.shape[1])

Feature matrix shape: (418381, 97)
Target shape: (418381,)
Number of features: 97


In [12]:
print("Categorical features:")
print(X.select_dtypes(include=["object"]).columns.tolist())

print("\nNumerical features:")
print(X.select_dtypes(exclude=["object"]).columns.tolist())

Categorical features:
['wd', 'station']

Numerical features:
['year', 'month', 'day', 'hour', 'PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM', 'day_of_week', 'is_weekend', 'PM2.5_lag_1h', 'PM2.5_lag_3h', 'PM2.5_lag_6h', 'PM10_lag_1h', 'PM10_lag_3h', 'PM10_lag_6h', 'SO2_lag_1h', 'SO2_lag_3h', 'SO2_lag_6h', 'NO2_lag_1h', 'NO2_lag_3h', 'NO2_lag_6h', 'CO_lag_1h', 'CO_lag_3h', 'CO_lag_6h', 'O3_lag_1h', 'O3_lag_3h', 'O3_lag_6h', 'PM2.5_rolling_mean_3h', 'PM2.5_rolling_max_3h', 'PM2.5_rolling_std_3h', 'PM2.5_rolling_mean_6h', 'PM2.5_rolling_max_6h', 'PM2.5_rolling_std_6h', 'PM10_rolling_mean_3h', 'PM10_rolling_max_3h', 'PM10_rolling_std_3h', 'PM10_rolling_mean_6h', 'PM10_rolling_max_6h', 'PM10_rolling_std_6h', 'SO2_rolling_mean_3h', 'SO2_rolling_max_3h', 'SO2_rolling_std_3h', 'SO2_rolling_mean_6h', 'SO2_rolling_max_6h', 'SO2_rolling_std_6h', 'NO2_rolling_mean_3h', 'NO2_rolling_max_3h', 'NO2_rolling_std_3h', 'NO2_rolling_mean_6h', 'NO2_rolling_max_6h', 'NO2_ro

C:\Users\HP\AppData\Local\Temp\ipykernel_16404\2682472949.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(X.select_dtypes(include=["object"]).columns.tolist())


### Findings

The labeled dataset contains **97 features** after excluding the target, record identifier, and datetime columns. The features include **95 numerical variables** and **2 categorical variables** (`wd` and `station`).


## 3. Temporal Data Split

The data is divided chronologically into model-training, validation, and final test periods.

The model-training period is used to train the XGBoost model, while the validation period is used for hyperparameter tuning and model selection. The final test period remains completely unseen until the final evaluation.

This chronological split prevents future information from being used during model development.


### 3.1 Define the periods

In [8]:
MODEL_TRAIN_END = "2014-12-31 23:00:00"
VALIDATION_START = "2015-01-01 00:00:00"
VALIDATION_END = "2015-12-31 23:00:00"
TEST_START = "2016-01-01 00:00:00"

model_train_mask = df["datetime"] <= MODEL_TRAIN_END

validation_mask = (
    (df["datetime"] >= VALIDATION_START)
    & (df["datetime"] <= VALIDATION_END)
)

test_mask = df["datetime"] >= TEST_START

### 3.2 Create the splits

In [9]:
model_train_df = df.loc[model_train_mask].copy()
validation_df = df.loc[validation_mask].copy()
test_df = df.loc[test_mask].copy()

print("Model training period:")
print(model_train_df["datetime"].min(), "to", model_train_df["datetime"].max())
print("Observations:", len(model_train_df))

print("\nValidation period:")
print(validation_df["datetime"].min(), "to", validation_df["datetime"].max())
print("Observations:", len(validation_df))

print("\nFinal test period:")
print(test_df["datetime"].min(), "to", test_df["datetime"].max())
print("Observations:", len(test_df))

Model training period:
2013-03-01 00:00:00 to 2014-12-31 23:00:00
Observations: 191665

Validation period:
2015-01-01 00:00:00 to 2015-12-31 23:00:00
Observations: 104816

Final test period:
2016-01-01 00:00:00 to 2017-02-28 17:00:00
Observations: 121900


### 3.3 Prepare X and y

In [12]:
X_model_train = model_train_df.drop(columns=EXCLUDED_COLUMNS)
y_model_train = model_train_df[TARGET]

X_validation = validation_df.drop(columns=EXCLUDED_COLUMNS)
y_validation = validation_df[TARGET]

X_test = test_df.drop(columns=EXCLUDED_COLUMNS)
y_test = test_df[TARGET]

print("X_model_train:", X_model_train.shape)
print("y_model_train:", y_model_train.shape)

print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_model_train: (191665, 97)
y_model_train: (191665,)
X_validation: (104816, 97)
y_validation: (104816,)
X_test: (121900, 97)
y_test: (121900,)


### 3.4 Verify temporal separation

In [13]:
print(
    "Training ends:",
    model_train_df["datetime"].max()
)

print(
    "Validation starts:",
    validation_df["datetime"].min()
)

print(
    "Validation ends:",
    validation_df["datetime"].max()
)

print(
    "Test starts:",
    test_df["datetime"].min()
)

Training ends: 2014-12-31 23:00:00
Validation starts: 2015-01-01 00:00:00
Validation ends: 2015-12-31 23:00:00
Test starts: 2016-01-01 00:00:00


### Findings

The data was split chronologically into **191,665 training observations**, **104,816 validation observations**, and **121,900 final test observations**.

The training, validation, and test periods are strictly separated in time, ensuring that future observations are not used during model development.


## 4. XGBoost Preprocessing

The preprocessing pipeline prepares the numerical and categorical features for XGBoost.

Missing numerical values are replaced using the median, while missing categorical values are replaced using the most frequent category. Categorical variables are one-hot encoded.

Numerical features are not standardized because XGBoost is a tree-based model and does not require feature scaling.


In [14]:
categorical_features = ["wd", "station"]

numerical_features = [
    col for col in X_model_train.columns
    if col not in categorical_features
]

print("Categorical features:", categorical_features)
print("Number of numerical features:", len(numerical_features))

Categorical features: ['wd', 'station']
Number of numerical features: 95


In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        ),
        (
            "numerical",
            SimpleImputer(strategy="median"),
            numerical_features
        )
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


## 5. Baseline XGBoost

A baseline XGBoost model is trained using the initial hyperparameter configuration selected during the model development stage.

The baseline provides a reference point for evaluating whether hyperparameter tuning improves model performance.

The model is trained only on the model-training period, while the validation period remains unseen during training.


In [16]:
baseline_xgb = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            XGBClassifier(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=6,
                random_state=42,
                n_jobs=-1,
                eval_metric="logloss"
            )
        )
    ]
)

print("Baseline XGBoost pipeline created successfully.")

Baseline XGBoost pipeline created successfully.


In [17]:
start_time = time.time()

baseline_xgb.fit(
    X_model_train,
    y_model_train
)

training_time = time.time() - start_time

print(f"Baseline XGBoost training time: {training_time:.2f} seconds")

Baseline XGBoost training time: 42.36 seconds


## 6. Baseline Evaluation

The baseline XGBoost model is evaluated on the held-out validation period.

The evaluation uses Accuracy, Precision, Recall, F1-score, ROC-AUC, and PR-AUC. Recall is particularly important for AirShift because missing a true deterioration event may reduce the usefulness of the early warning system.

The validation set is not used to train the baseline model.


In [18]:
y_validation_pred = baseline_xgb.predict(X_validation)
y_validation_prob = baseline_xgb.predict_proba(X_validation)[:, 1]

print("Validation predictions generated successfully.")

Validation predictions generated successfully.


In [22]:
baseline_metrics = {
    "Accuracy": accuracy_score(y_validation, y_validation_pred),
    "Precision": precision_score(y_validation, y_validation_pred),
    "Recall": recall_score(y_validation, y_validation_pred),
    "F1": f1_score(y_validation, y_validation_pred),
    "ROC-AUC": roc_auc_score(y_validation, y_validation_prob),
    "PR-AUC": average_precision_score(y_validation, y_validation_prob)
}

baseline_metrics

{'Accuracy': 0.7253472752251565,
 'Precision': 0.7117611212675198,
 'Recall': 0.7059107941496434,
 'F1': 0.7088238863939799,
 'ROC-AUC': 0.8073744433092597,
 'PR-AUC': 0.7981817459134355}

In [23]:
baseline_results = pd.DataFrame(
    [baseline_metrics],
    index=["Baseline XGBoost"]
)

baseline_results.round(4)

,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
Baseline XGBoost,0.7253,0.7118,0.7059,0.7088,0.8074,0.7982


### Findings

| Metric    | Baseline XGBoost |
| --------- | ---------------: |
| Accuracy  |           0.7253 |
| Precision |           0.7118 |
| Recall    |           0.7059 |
| F1-score  |           0.7088 |
| ROC-AUC   |           0.8074 |
| PR-AUC    |           0.7982 |

The baseline model provides a reference point for evaluating the performance improvements from hyperparameter tuning.


## 7. Hyperparameter Tuning

Hyperparameter tuning is used to identify an improved XGBoost configuration for the AirShift early warning task.

The tuning process uses **TimeSeriesSplit** on the model-training period only. This preserves the chronological order of the observations and prevents future observations from being used to evaluate earlier training periods.

The validation period remains completely unseen during hyperparameter search and is used afterward to compare the baseline and tuned models.

The search focuses on parameters that control model complexity, learning rate, number of trees, and feature and sample subsampling.


In [24]:
tscv = TimeSeriesSplit(n_splits=3)

print("Number of time-series splits:", tscv.n_splits)

Number of time-series splits: 3


In [25]:
# Define the hyperparameter search space
xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            XGBClassifier(
                random_state=42,
                n_jobs=-1,
                eval_metric="logloss"
            )
        )
    ]
)

print("XGBoost tuning pipeline created successfully.")

XGBoost tuning pipeline created successfully.


In [26]:
# Define the hyperparameter search space
param_distributions = {
    "model__n_estimators": [100, 200, 300, 400],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__max_depth": [3, 4, 5, 6, 8],
    "model__min_child_weight": [1, 3, 5],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0]
}

print("Number of hyperparameter groups:", len(param_distributions))

Number of hyperparameter groups: 6


In [27]:
# Randomized Search
random_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_distributions,
    n_iter=15,
    scoring="average_precision",
    cv=tscv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("RandomizedSearchCV configured successfully.")

RandomizedSearchCV configured successfully.


In [ ]:
# Run the tuning

start_time = time.time()

random_search.fit(
    X_model_train,
    y_model_train
)

tuning_time = time.time() - start_time

print(f"Tuning time: {tuning_time:.2f} seconds")

Fitting 3 folds for each of 15 candidates, totalling 45 fits
Tuning time: 1194.69 seconds


In [31]:
# Best Hyperparameters

print("Best hyperparameters:")
print(random_search.best_params_)

print("\nBest cross-validation PR-AUC:")
print(round(random_search.best_score_, 4))

Best hyperparameters:
{'model__subsample': 0.7, 'model__n_estimators': 400, 'model__min_child_weight': 3, 'model__max_depth': 6, 'model__learning_rate': 0.1, 'model__colsample_bytree': 1.0}

Best cross-validation PR-AUC:
0.8324


### Findings

| Item                    | Result                    |
| ----------------------- | ------------------------- |
| Cross-validation        | TimeSeriesSplit (3 folds) |
| Configurations tested   | 15                        |
| Total model fits        | 45                        |
| Tuning metric           | PR-AUC                    |
| Best CV PR-AUC          | **0.8324**                |
| Best `n_estimators`     | 400                       |
| Best `learning_rate`    | 0.10                      |
| Best `max_depth`        | 6                         |
| Best `min_child_weight` | 3                         |
| Best `subsample`        | 0.70                      |
| Best `colsample_bytree` | 1.00                      |
| Tuning time             | 517.46 seconds            |

The best configuration achieved a cross-validation PR-AUC of **0.8324** and will be evaluated on the held-out validation period before final model selection.


## 8. Tuned Model Evaluation

The best XGBoost configuration identified during hyperparameter tuning is evaluated on the held-out validation period.

The validation data from 2015 was not used during hyperparameter search, allowing an independent comparison between the baseline and tuned models.

The tuned model is evaluated using Accuracy, Precision, Recall, F1-score, ROC-AUC, and PR-AUC.


### 8.1 Generate Predictions

In [32]:
tuned_xgb = random_search.best_estimator_

y_validation_tuned_pred = tuned_xgb.predict(X_validation)
y_validation_tuned_prob = tuned_xgb.predict_proba(X_validation)[:, 1]

print("Tuned XGBoost validation predictions generated successfully.")

Tuned XGBoost validation predictions generated successfully.


### 8.2 Calculate Metrics

In [33]:
tuned_metrics = {
    "Accuracy": accuracy_score(
        y_validation,
        y_validation_tuned_pred
    ),
    "Precision": precision_score(
        y_validation,
        y_validation_tuned_pred
    ),
    "Recall": recall_score(
        y_validation,
        y_validation_tuned_pred
    ),
    "F1": f1_score(
        y_validation,
        y_validation_tuned_pred
    ),
    "ROC-AUC": roc_auc_score(
        y_validation,
        y_validation_tuned_prob
    ),
    "PR-AUC": average_precision_score(
        y_validation,
        y_validation_tuned_prob
    )
}

tuned_metrics

{'Accuracy': 0.7279136773011754,
 'Precision': 0.7124519646701406,
 'Recall': 0.7133849067246867,
 'F1': 0.7129181304798623,
 'ROC-AUC': 0.8102345162893226,
 'PR-AUC': 0.8015125344219024}

### 8.3 Display Results

In [34]:
tuned_results = pd.DataFrame(
    [tuned_metrics],
    index=["Tuned XGBoost"]
)

tuned_results.round(4)

,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
Tuned XGBoost,0.7279,0.7125,0.7134,0.7129,0.8102,0.8015


### Findings

| Metric    | Tuned XGBoost |
| --------- | ------------: |
| Accuracy  |        0.7279 |
| Precision |        0.7125 |
| Recall    |        0.7134 |
| F1-score  |        0.7129 |
| ROC-AUC   |        0.8102 |
| PR-AUC    |        0.8015 |

The tuned XGBoost model achieved a Recall of **0.7134**, indicating that it detected approximately 71% of the deterioration events in the validation period.

The model achieved a **ROC-AUC of 0.8102** and a **PR-AUC of 0.8015**, indicating good ability to distinguish between deterioration and non-deterioration events.

These results will be compared with the baseline XGBoost model in the next section to determine whether hyperparameter tuning improved the model's performance.


## 9. Baseline vs Tuned

The baseline and tuned XGBoost models are compared on the same held-out validation period.

This comparison determines whether hyperparameter tuning improves the model's ability to detect future deterioration events.


In [35]:
comparison_results = pd.concat(
    [
        baseline_results,
        tuned_results
    ]
)

comparison_results.round(4)

,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
Baseline XGBoost,0.7253,0.7118,0.7059,0.7088,0.8074,0.7982
Tuned XGBoost,0.7279,0.7125,0.7134,0.7129,0.8102,0.8015


In [36]:
improvement = (
    comparison_results.loc["Tuned XGBoost"]
    - comparison_results.loc["Baseline XGBoost"]
)

print("Improvement after tuning:")
print(improvement.round(4))

Improvement after tuning:
Accuracy     0.0026
Precision    0.0007
Recall       0.0075
F1           0.0041
ROC-AUC      0.0029
PR-AUC       0.0033
dtype: float64


### Findings

The tuned XGBoost model improved **all evaluation metrics** compared with the baseline model.

The largest improvement was in **Recall**, increasing from **0.7059 to 0.7134**, which is particularly important for the AirShift early warning task.

Overall, hyperparameter tuning provided a modest but consistent improvement in model performance.


In [37]:
# Create output directory
FIGURES_DIR = Path("../reports/figures/07_XGBoost")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC",
    "PR-AUC"
]

baseline_values = comparison_results.loc["Baseline XGBoost", metrics]
tuned_values = comparison_results.loc["Tuned XGBoost", metrics]

x = np.arange(len(metrics))
width = 0.35

plt.figure(figsize=(10, 5))

plt.bar(
    x - width / 2,
    baseline_values,
    width,
    label="Baseline XGBoost",
    color="#78909C"
)

plt.bar(
    x + width / 2,
    tuned_values,
    width,
    label="Tuned XGBoost",
    color="#607D8B"
)

plt.xticks(x, metrics)
plt.ylabel("Score")
plt.title("Baseline vs Tuned XGBoost")
plt.ylim(0.65, 0.85)

plt.legend()
plt.tight_layout()

# Save figure
output_path = FIGURES_DIR / "09_baseline_vs_tuned.png"
plt.savefig(output_path, dpi=300, bbox_inches="tight")

plt.show()

print(f"Figure saved to: {output_path}")


KeyboardInterrupt



KeyboardInterrupt: 

## 10. Select Final Configuration

Based on the validation results, the tuned XGBoost model is selected as the final model configuration.

The tuned model improved all evaluated metrics compared with the baseline model, with the largest improvement in Recall.

The selected configuration will be retrained using the combined training and validation data before final evaluation on the unseen test period.


In [39]:
final_params = random_search.best_params_

print("Selected final XGBoost configuration:")

for param, value in final_params.items():
    print(f"{param}: {value}")

Selected final XGBoost configuration:
model__subsample: 0.7
model__n_estimators: 400
model__min_child_weight: 3
model__max_depth: 6
model__learning_rate: 0.1
model__colsample_bytree: 1.0


### Findings

The tuned XGBoost configuration was selected as the final model configuration based on its improved validation performance.

The selected model will be retrained using the combined training and validation data before final testing.


## 11. Prepare Final Model for Test

The selected XGBoost configuration is retrained using the combined model-training and validation periods.

The final test period remains completely unseen during training and will only be used for the final model evaluation.


In [19]:
X_train_final = pd.concat(
    [X_model_train, X_validation],
    axis=0
)

y_train_final = pd.concat(
    [y_model_train, y_validation],
    axis=0
)

print("Final training features:", X_train_final.shape)
print("Final training target:", y_train_final.shape)

Final training features: (296481, 97)
Final training target: (296481,)


### Final Model

In [22]:
final_xgb = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            XGBClassifier(
                n_estimators=400,
                learning_rate=0.1,
                max_depth=6,
                min_child_weight=3,
                subsample=0.7,
                colsample_bytree=1.0,
                random_state=42,
                n_jobs=2,
                eval_metric="logloss"
            )
        )
    ]
)

In [23]:
# Final XGBoost training

print("Starting final XGBoost training...")
print("Training data:", X_train_final.shape)
print("Target:", y_train_final.shape)

final_xgb.fit(X_train_final, y_train_final)

print("Final XGBoost training completed successfully.")

Starting final XGBoost training...
Training data: (296481, 97)
Target: (296481,)
Final XGBoost training completed successfully.


### Save Final Model

The final XGBoost model is saved for reproducibility and later evaluation on the unseen test period.


In [24]:
from pathlib import Path
import joblib

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "xgboost_final.joblib"

# Verify that XGBoost is fitted before saving
is_fitted = hasattr(final_xgb.named_steps["model"], "n_features_in_")

print("XGBoost fitted:", is_fitted)

if is_fitted:
    joblib.dump(final_xgb, MODEL_PATH)

    print(f"Final model saved to: {MODEL_PATH}")
    print(f"Model file exists: {MODEL_PATH.exists()}")
    print(f"Model file size: {MODEL_PATH.stat().st_size / 1024**2:.2f} MB")
else:
    print("Model was NOT saved because XGBoost is not fitted.")

XGBoost fitted: True
Final model saved to: ..\models\xgboost_final.joblib
Model file exists: True
Model file size: 1.74 MB
